# Greedy Algorithm

In [ ]:
from itertools import combinations
import time

In [ ]:
items = [
    {"weight": 10, "value": 60},
    {"weight": 20, "value": 100},
    {"weight": 30, "value": 120},
    {"weight": 25, "value": 80},
    {"weight": 15, "value": 40},
    {"weight": 5, "value": 30},
    {"weight": 22, "value": 50},
    {"weight": 18, "value": 70},
    {"weight": 11, "value": 45},
    {"weight": 9, "value": 55},
    {"weight": 14, "value": 65},
    {"weight": 12, "value": 60},
    {"weight": 7, "value": 20},
    {"weight": 8, "value": 25},
    {"weight": 13, "value": 55}
]

capacity = 50

## Greedy Algorithm with Greedy criteria as Value to Weight ratio

In [ ]:
def get_ratio(item):
    return item["value"] / item["weight"]

In [ ]:
def knapsack_greedy(items, capacity):

    # Call the function 'get_ratio' and sort the values in descending order
    items_sorted = sorted(items, key=get_ratio, reverse=True)

    total_value = 0
    total_weight = 0
    chosen_items = []

    # Pick items with the best value/weight ratio first, as long as they fit
    for item in items_sorted:
        if total_weight + item["weight"] <= capacity:
            chosen_items.append(item)
            total_weight += item["weight"]
            total_value += item["value"]

    return total_value, chosen_items, total_weight

In [ ]:
# Call Greedy solution and print Total value, Chosen items, Total weight and What weight is remaining
greedy_value, greedy_items, greedy_weight = knapsack_greedy(items, capacity)

print("Greedy Solution")
print("Total value     :", greedy_value)
print("Chosen items    :")
for item in greedy_items:
    print("   ", item, "ratio =", round(get_ratio(item), 2))
print("Total weight    :", greedy_weight)
print("Remaining weight:", capacity - greedy_weight)

## Greedy Algorithm with Bruteforce approach

In [ ]:
def knapsack_brute_force(items, capacity):
    n = len(items)
    best_value = 0
    best_combination = None
    used_capacity = None

    for r in range(n + 1):
        for combination in combinations(items, r):
            total_weight = sum(item["weight"] for item in combination)
            total_value = sum(item["value"] for item in combination)

            if total_weight <= capacity and total_value > best_value:
                best_value = total_value
                best_combination = combination
                used_capacity = total_weight

    return best_value, best_combination, used_capacity

In [ ]:
# Call brute force solution and print Total value, Chosen items, Total weight and What weight is remaining
bf_value, bf_items, bf_weight = knapsack_brute_force(items, capacity)

print("Brute Force Solution")
print("Total value     :", bf_value)
print("Chosen items    :")
for item in bf_items:
    print("   ", item)
print("Total weight    :", bf_weight)
print("Remaining weight:", capacity - bf_weight)

# Check the time complexity of these 2 algorithms

## Track how much time each of these algorithms take for the sample data

In [ ]:
# Time the Greedy algorithm
start = time.perf_counter()
knapsack_greedy(items, capacity)
greedy_time = time.perf_counter() - start

print(f"Greedy time      : {greedy_time:.6f} seconds")

In [ ]:
# Time the Brute Force algorithm
start = time.perf_counter()
knapsack_brute_force(items, capacity)
bf_time = time.perf_counter() - start

print(f"Brute Force time : {bf_time:.6f} seconds")

In [ ]:
# Compare the two
print(f"Brute Force is ~{bf_time / greedy_time:,.0f}x slower than Greedy")
print(f"Greedy value = {greedy_value}, Brute Force (optimal) value = {bf_value}")

# Greedy  -> O(n log n)  (dominated by sorting)
# Brute   -> O(2^n * n)  (checks every subset; 2^15 = 32,768 subsets here)
# Greedy is fast but NOT guaranteed optimal for 0/1 knapsack; brute force is always optimal but explodes as n grows.

## Greedy Algorithm with Recursive approach

Convert Brute Force into Recursion by changing how we generate the subsets: instead of building complete combinations upfront with itertools.combinations, we make one "Yes/No" decision per item and let the function call itself to handle the remaining items.

In [ ]:
def knapsack_recursive(items, capacity, n=None):
    if n is None:
        n = len(items)

    # Base case: no items left or backpack has 0 capacity
    if n == 0 or capacity == 0:
        return 0

    current_item = items[n - 1]

    # If the current item is heavier than the remaining capacity, skip it
    if current_item["weight"] > capacity:
        return knapsack_recursive(items, capacity, n - 1)

    # Include the current item
    include = current_item["value"] + knapsack_recursive(
        items, capacity - current_item["weight"], n - 1)

    # Exclude the current item
    exclude = knapsack_recursive(items, capacity, n - 1)

    return max(include, exclude)

In [ ]:
# Call recursive solution and print the value
rec_value = knapsack_recursive(items, capacity)
print("Recursive Solution - Total value:", rec_value)

In [ ]:
# Check how much time it takes to execute recursive solution?
start = time.perf_counter()
knapsack_recursive(items, capacity)
rec_time = time.perf_counter() - start

print(f"Recursive time   : {rec_time:.6f} seconds")

## Knapsack_dynamic programming with Tabulation: Removes recursion completely and fills the same grid iteratively from bottom to top using for loops — (O(n * W)).


In [ ]:
def knapsack_dp(items, capacity):
    n = len(items)

    # Create an (n + 1) x (capacity + 1) 2D table initialized with 0
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]

    # Start from 1 since row 0 (0 items) and column 0 (0 capacity) stay 0
    for i in range(1, n + 1):
        current_item = items[i - 1]

        for w in range(1, capacity + 1):
            # Check if the current item fits in the current capacity 'w'
            if current_item["weight"] <= w:
                include = current_item["value"] + dp[i - 1][w - current_item["weight"]]
                exclude = dp[i - 1][w]

                dp[i][w] = max(include, exclude)
            else:
                # Item is too heavy: copy the value directly from the row above
                dp[i][w] = dp[i - 1][w]

    return dp[n][capacity]

In [ ]:
# Call dp solution and print the value
dp_value = knapsack_dp(items, capacity)
print("DP Solution - Total value:", dp_value)

In [ ]:
# Check how much time it takes to execute dp solution?
start = time.perf_counter()
knapsack_dp(items, capacity)
dp_time = time.perf_counter() - start

print(f"DP time          : {dp_time:.6f} seconds")

In [ ]:
# Summary: compare all four approaches
print(f"{'Approach':<12}{'Value':>8}{'Time (s)':>14}   Complexity")
print("-" * 52)
print(f"{'Greedy':<12}{greedy_value:>8}{greedy_time:>14.6f}   O(n log n)  - not always optimal")
print(f"{'Brute Force':<12}{bf_value:>8}{bf_time:>14.6f}   O(2^n * n)")
print(f"{'Recursive':<12}{rec_value:>8}{rec_time:>14.6f}   O(2^n)")
print(f"{'DP':<12}{dp_value:>8}{dp_time:>14.6f}   O(n * W)")